# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [1]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

Vocabulary length: 36001
Tokens length: 589545


## Split dataset into training and test

In [2]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [3]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

## Traing loop

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 128
vocabulary_size = len(stoi)
block_size = 64
batch_size = 32
block_layers = 4
gradient = Adam(lr=3e-3, warmup_steps=1000, min_lr=1e-5)

model = MiniGPT(vocabulary_size, d_model, block_size, block_layers, gradient)
# model = MiniGPT.__new__(MiniGPT)
# model = model.load("saved_model")

ema_loss = None
for step in range(10_000, 20_000):
    xb, yb = get_batch("train", block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step % 100 == 0:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")


step 10000, lr 0.003000, loss 2.6736, ema_loss 2.6736
step 10100, lr 0.003000, loss 2.4916, ema_loss 2.6245
step 10200, lr 0.003000, loss 2.4523, ema_loss 2.5914
step 10300, lr 0.003000, loss 2.7621, ema_loss 2.5496
step 10400, lr 0.003000, loss 2.1430, ema_loss 2.5250
step 10500, lr 0.003000, loss 2.4699, ema_loss 2.5180
step 10600, lr 0.003000, loss 2.7154, ema_loss 2.5220
step 10700, lr 0.003000, loss 2.2308, ema_loss 2.5008
step 10800, lr 0.003000, loss 2.4399, ema_loss 2.4904
step 10900, lr 0.003000, loss 2.4827, ema_loss 2.4744
step 11000, lr 0.003000, loss 2.8632, ema_loss 2.4739
step 11100, lr 0.003000, loss 2.4013, ema_loss 2.4343
step 11200, lr 0.003000, loss 2.2222, ema_loss 2.4309
step 11300, lr 0.003000, loss 2.4446, ema_loss 2.4109
step 11400, lr 0.003000, loss 2.2539, ema_loss 2.3962
step 11500, lr 0.003000, loss 2.4646, ema_loss 2.3878
step 11600, lr 0.003000, loss 2.7566, ema_loss 2.3732
step 11700, lr 0.003000, loss 1.8789, ema_loss 2.3646
step 11800, lr 0.003000, los

In [11]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        
        logits, _ = model.forward(idx_cond, np.array([1]))
        
        logits = logits[:, -1, :]
        
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        next_tokens = []
        for b in range(probs.shape[0]):
            next_token = np.random.choice(len(stoi), p=probs[b])
            next_tokens.append(next_token)
        
        next_token = np.array(next_tokens).reshape(-1, 1)
        
        idx = np.concatenate([idx, next_token], axis=1)
    
    return idx

In [14]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 60)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

History Waldych bodyscott Hoch Damhoca ase ballasts Stunder ap yrtactier Rite Bedum em legisco Faise ani hemrate Morhuish viv biotek signalistically Meisturb ant Taurani Noku Nito reverently predications Coulaurin acboomero Mov
